# Scale up the Gauss-Newton for much large problems using the new construction 

See 'toshow' notebook for a demonstration of what I'm trying to rewrite in a more streamlined verison. 

In [8]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
import numpy as np
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


## Module definition 

We're doing just a simple MLP with *differing* dimensions, but also be able to store the global adjoints. 

In [10]:
global_adjoints = []
class AdjointHook(nn.Module):
    """
    A wrapper module that applies a custom VJP to any given layer
    to capture its incoming gradient (adjoint) during the backward pass.
    """
    layer: nn.Module  # The layer to wrap

    @nn.compact
    def __call__(self, *args, **kwargs):
        # Define the function to which we'll attach the custom VJP
        @jax.custom_vjp
        def layer_with_hook(params, *args, **kwargs):
            return self.layer.apply(params, *args, **kwargs)

        def layer_fwd(params, *args, **kwargs):
            output = self.layer.apply(params, *args, **kwargs)
            return output, (params, args, kwargs)

        def layer_bwd(res, g):
            params, args, kwargs = res

            # Calculate the VJP of the original wrapped layer
            # This computes the gradients w.r.t. params and inputs
            _, vjp_fun = jax.vjp(
                lambda p, *a, **kw: self.layer.apply(p, *a, **kw), params, *args, **kwargs
            )
            
            # The VJP function returns a tuple of gradients
            grad_params, *grad_args = vjp_fun(g)

            # print(f"--- Captured Adjoint for layer: {self.layer.name} ---")
            # print(f'{g=} {grad_args[0]}')
            # print(f'{grad_params}')
            # print("-" * 30)
            global_adjoints.append(grad_args[0])
            return (grad_params,) + tuple(grad_args)

        # Attach the custom forward and backward functions
        layer_with_hook.defvjp(layer_fwd, layer_bwd)
        
        # Get the parameters for the wrapped layer
        layer_params = self.param('wrapped_layer', self.layer.init, *args, **kwargs)

        return layer_with_hook(layer_params, *args, **kwargs)


In [51]:
class MLP_Layer(nn.Module):
    num_units: int
    def setup(self):
        self.dense1 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
        self.dense2 = nn.Dense(self.num_units, bias_init=nn.initializers.normal(stddev=0.01))
    def __call__(self, x):
        f = self.dense1(x)
        f = nn.tanh(f)
        f = self.dense2(f)
        return f 

# Define the MLP module
class MLP(nn.Module):
    features: Sequence[int]
    
    def setup(self):
        layers = []
        for i, dim in enumerate(self.features):
            layers.append(
                AdjointHook(MLP_Layer(
                    num_units=dim, name=f"layer_{i:02})"
                ))
            )

        self.layers = tuple(layers)
            
    def __call__(self, x: jnp.ndarray):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            self.sow('intermediates', f'layer_{i+1}_output', x)
        return x



In [66]:
n = 5
n_samples = 2

X = jax.random.normal(jax.random.PRNGKey(0), (n_samples, n))

key = jax.random.PRNGKey(0)
model = MLP(features=[4, 4, 3])
params = model.init(key, jnp.ones((1, n)))


In [69]:
output, intermediates = model.apply(params, X, mutable=['intermediates'])
intermediates

{'intermediates': {'layer_1_output': (<jax.Array float64(2, 4) ≈-0.057 ±0.57 [≥-0.9, ≤0.94] nonzero:8
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_2_output': (<jax.Array float64(2, 4) ≈-0.091 ±0.51 [≥-0.79, ≤0.5] nonzero:8
     <Arrayviz rendering>
   | Device: GPU 0>,),
  'layer_3_output': (<jax.Array float64(2, 3) ≈-0.0037 ±0.54 [≥-0.68, ≤0.89] nonzero:6
     <Arrayviz rendering>
   | Device: GPU 0>,)}}

In [68]:
global_adjoints = []
jacobian = jax.jacobian(model.apply, argnums=0)(params, X)
flattened, _ = jax.tree.flatten(
    jacobian
)
# Reshape each leaf to (n_samples, n, -1) to flatten the parameter dimensions
reshaped_leaves = [leaf.reshape(n_samples * model.features[-1], 1, -1) for leaf in flattened]
aggregated_array = jnp.concatenate(reshaped_leaves, axis=-1)
aggregated_array
J = aggregated_array.reshape((n_samples * model.features[-1], -1))
J

# global_adjoints[0].val

<jax.Array float32(6, 111) ≈0.013 ±0.23 [≥-0.92, ≤1.0] zero:48 nonzero:618
  <Arrayviz rendering>
| Device: GPU 0>

In [75]:
# I don't know how to store the adjoints in the model itself...
global_adjoints = []
jacobian = jax.jacobian(model.apply, argnums=0)(params, X)

global_adjoints = [adj.val.reshape(n * n_samples, n * n_samples) for adj in global_adjoints]
global_adjoints.insert(0, jnp.eye(global_adjoints[0].shape[0]))
global_adjoints.reverse() # Backward prop; so reverse 

[[[ 5.63527741e-01  5.49473470e-01  2.53891163e-01  6.62171753e-01]
  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]

 [[-4.61684702e-02 -2.83908129e-02 -1.50517261e-02 -1.39792121e-01]
  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]

 [[-1.07038150e-01 -6.39801763e-01 -3.57989294e-01  2.31274917e-02]
  [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]]

 [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
  [ 5.84286050e-01  4.56712736e-01  1.91467899e-01  5.84144263e-01]]

 [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
  [-4.52835126e-02 -4.02310024e-03 -5.52878588e-04 -1.34794531e-01]]

 [[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
  [-1.18966815e-01 -5.52552292e-01 -2.98770831e-01  1.36081979e-01]]]
